###Ingest circuits.csv file
        1. Read the file using spark dataframe reader API
        2. Add Metadata Columns
            - Source File
            - Ingestion Timestamp
        3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
%python
landing_folder_path
catalog_name

In [0]:
%python
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
%python
source_file

####Step 1 - Read the CSV file using the dataframe reader API

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType()),
    ])


In [0]:
%python
circuits_df = (
    spark.read
        .format('csv')
        .option('header', True)
#        .option('inferSchema', True)
        .option('mode', 'FAILFAST')
        .schema(circuits_schema)
        .load(source_file)
)

In [0]:
%python
display(circuits_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
%python
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
%python

display(circuits_final_df)

#### Step 3 - Write to bronze delta table

In [0]:
%python
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%python
display(spark.table(table_name))